# Chloris data pipeline (steps 1–3)
Medicinal plants native to Greece, built from open data. Each cell calls the same functions as `python -m chloris.run`,
so the notebook and the command line give the same files. Run it from the `pipeline/` folder.

Sources: WCVP (Kew, data behind POWO), World Checklist of Useful Plant Species 2020 (Diazgranados et al., Kew, CC BY 4.0, DOI 10.5063/F1CV4G34), GBIF backbone and Red List categories. Later cells (occurrences, cells, scores) are added in Phase 2.

**Raw inputs** (not in git): `raw/wcvp/` (scripted download of `wcvp.zip`), `raw/wcups/` (KNB package; only a PDF). The GBIF zip is also never committed.

In [1]:
import json, pandas as pd
from chloris import config, wcvp, wcups, gbif, flags
pd.set_option("display.width", 160, "display.max_colwidth", 60)
config.ensure_dirs()
print("scoring config:", config.load_scoring().version)

scoring config: v0-draft


## Step 1 — Native Greek species (WCVP)
Accepted species with a native, non-extinct, non-doubtful record in GRC, KRI or EAI. Accepted subspecies/varieties are folded into their parent species. A species is *endemic* when all its native areas are Greek.

In [2]:
greek = wcvp.run()
print(f"\nGreek native species: {len(greek):,}   endemic to Greece: {int(greek.endemic_gr.sum()):,}")

names: 1,448,984 rows; distribution: 1,995,338 rows
columns: ['plant_name_id', 'ipni_id', 'taxon_rank', 'taxon_status', 'family', 'genus', 'taxon_name', 'taxon_authors', 'accepted_plant_name_id', 'parent_plant_name_id', 'powo_id'] ['plant_name_id', 'area_code_l3', 'introduced', 'extinct', 'location_doubtful']


accepted species: 365,813; accepted infraspecific mapped to a species: 54,295


Greek native species: 6,081; endemic to Greece: 1,293


  plant_name_id     ipni_id     powo_id           taxon_name taxon_authors     family   genus      greek_areas  endemic_gr
0        107386    448797-1    448797-1  Lamium amplexicaule            L.  Lamiaceae  Lamium  [EAI, GRC, KRI]       False
1        107402    448803-1    448803-1       Lamium bifidum       Cirillo  Lamiaceae  Lamium       [GRC, KRI]       False
2        107452  60452701-2  60452701-2   Lamium galeobdolon       (L.) L.  Lamiaceae  Lamium            [GRC]       False
3        107458    448848-1    448848-1    Lamium garganicum            L.  Lamiaceae  Lamium  [EAI, GRC, KRI]       False
4        107493    448866-1    448866-1      Lamium hybridum         Vill.  Lamiaceae  Lamium            [GRC]       False

Greek native species: 6,081   endemic to Greece: 1,293


## Step 2 — Medicinal filter (WCUPS)
The KNB download is a 689-page PDF, so the rows are extracted from its text (method and limits: `WCUPS_EXTRACTION.md`). **Legend (PDF p. 4): ME = Medicines (human and veterinary).** Join to WCVP by IPNI id first, then by name.

In [3]:
med = wcups.run()

WCUPS rows parsed: 40,293 (dataset describes ~40,292); with ME: 26,663 (dataset says 26,662); duplicate IPNI ids: 58
rows with problems: 1,272 -> wcups_parse_issues.csv
problems
truncated_no_refs    1272


all WCUPS rows -> WCVP: by IPNI id 40,169; name only 69; unmatched 55


WCUPS rows hitting a Greek native: by IPNI id 1820, name only 207
featured species: 12; Greek natives in WCVP: 12; added as curated (WCUPS missed): 0; featured names NOT a Greek native in WCVP (check spelling/synonym): []
{
  "wcups_rows": 40293,
  "wcups_with_ME": 26663,
  "greek_natives": 6081,
  "greek_with_any_wcups_row": 1847,
  "greek_medicinal": 1386,
  "medicinal_by_ipni": 1336,
  "medicinal_by_name_only": 50,
  "parse_issue_rows": 1272,
  "curated_added": 0,
  "medicinal_final": 1386
}


In [4]:
rows = pd.read_parquet(config.WORK_DIR / "wcups_rows.parquet")
issues = pd.read_csv(config.WORK_DIR / "wcups_parse_issues.csv")
print(f"WCUPS rows parsed: {len(rows):,} (dataset describes 40,292)   with ME: {int(rows.medicinal.sum()):,} (dataset states 26,662)")
print("rows flagged by the parser:", issues.problems.value_counts().to_dict())
print("ME among flagged rows:", int(rows[rows.truncated].medicinal.sum()), "of", int(rows.truncated.sum()))

WCUPS rows parsed: 40,293 (dataset describes 40,292)   with ME: 26,663 (dataset states 26,662)
rows flagged by the parser: {'truncated_no_refs': 1272}
ME among flagged rows: 0 of 1272


In [5]:
# Spot checks: well-known Greek medicinal plants
for n in ["Hypericum perforatum", "Origanum dictamnus", "Salvia fruticosa", "Crocus sativus", "Sideritis scardica", "Thymus vulgaris"]:
    in_greek = (greek.taxon_name == n).any(); r = rows[rows.binomial == n.lower()]
    print(f"{n:22} Greek native: {in_greek!s:5}  WCUPS codes: {list(r.codes.iloc[0]) if len(r) else None}  in final list: {(med.taxon_name == n).any()}")

Hypericum perforatum   Greek native: True   WCUPS codes: ['MA', 'ME', 'PO']  in final list: True
Origanum dictamnus     Greek native: True   WCUPS codes: ['HF', 'ME']  in final list: True
Salvia fruticosa       Greek native: True   WCUPS codes: ['HF', 'MA', 'ME', 'PO']  in final list: True
Crocus sativus         Greek native: True   WCUPS codes: ['EU', 'HF', 'MA', 'ME']  in final list: True
Sideritis scardica     Greek native: True   WCUPS codes: ['HF', 'ME']  in final list: True
Thymus vulgaris        Greek native: False  WCUPS codes: ['EU', 'HF', 'MA', 'ME', 'PO']  in final list: False


In [6]:
print(f"Medicinal species native to Greece: {len(med):,}")
print(med.match.value_counts().rename("matched by").to_frame().T.to_string())
print("\nFirst 20 species:")
med.sort_values(["family", "taxon_name"]).head(20)[["taxon_name", "taxon_authors", "family", "endemic_gr", "match", "medicinal_source"]].reset_index(drop=True)

Medicinal species native to Greece: 1,386
match       ipni  name
matched by  1336    50

First 20 species:


,taxon_name,taxon_authors,family,endemic_gr,match,medicinal_source
0,Acanthus hirsutus,Boiss.,Acanthaceae,False,ipni,wcups
1,Acanthus mollis,L.,Acanthaceae,False,ipni,wcups
2,Mesembryanthemum nodiflorum,L.,Aizoaceae,False,ipni,wcups
3,Alisma lanceolatum,With.,Alismataceae,False,ipni,wcups
4,Alisma plantago-aquatica,L.,Alismataceae,False,ipni,wcups
5,Sagittaria sagittifolia,L.,Alismataceae,False,ipni,wcups
6,Liquidambar orientalis,Mill.,Altingiaceae,False,ipni,wcups
7,Amaranthus graecizans,L.,Amaranthaceae,False,ipni,wcups
8,Arthrocaulon macrostachyum,(Moric.) Piirainen & G.Kadereit,Amaranthaceae,False,ipni,wcups
9,Atriplex halimus,L.,Amaranthaceae,False,ipni,wcups


## Step 3 — GBIF backbone keys, IUCN category, vernacular names
Species names are matched to the GBIF backbone (species rank; exact, or fuzzy with confidence ≥ 95). Every API response is cached in `work/cache/`.
**IUCN `NE` is shown as “Not assessed” and is never treated as low risk.**

In [7]:
keys = gbif.match_keys()
keys[~keys.accepted_key.notna()][["taxon_name", "match_type", "rank", "gbif_name"]].reset_index(drop=True)

species: 1386; matched EXACT: 1369; FUZZY: 6; HIGHERRANK/NONE: 11; usable keys: 1375; synonym-status matches: 34; distinct keys: 1373


,taxon_name,match_type,rank,gbif_name
0,Valeriana locusta,HIGHERRANK,FAMILY,Caprifoliaceae
1,Ranunculus minimus,HIGHERRANK,FAMILY,Ranunculaceae
2,Anchusa aegyptiaca,HIGHERRANK,FAMILY,Boraginaceae
3,Epilobium angustifolium,HIGHERRANK,GENUS,Chamaenerion Seguier
4,Taraxacum sect. Taraxacum,HIGHERRANK,SPECIES,Taraxacum spec F.H.Wigg.
5,Alchemilla arvensis,HIGHERRANK,FAMILY,Rosaceae
6,Rubus glandulosus,HIGHERRANK,GENUS,Rubus L.
7,Blechnum spicant,HIGHERRANK,GENUS,Struthiopteris Scop.
8,Mutarda arvensis,HIGHERRANK,GENUS,Mutarda Bernh.
9,Elaeoselinum garganicum,HIGHERRANK,GENUS,Elaeoselinum W.D.J.Koch ex DC.


In [8]:
info = gbif.iucn_and_names()
info["iucn_label"] = info.iucn.map(lambda c: config.IUCN_LABELS[c]["en"])
info.iucn_label.value_counts().to_frame("species")

IUCN counts:
 iucn
NE    1029
LC     330
DD      12
NT       4
VU       3
EN       1 
Greek name: 230; English name: 1256


,species
iucn_label,
Not assessed,1029
Least concern,330
Data deficient,12
Near threatened,4
Vulnerable,3
Endangered,1


### Data-quality and review flags
`data_quality` flags describe the WCVP→GBIF link (synonym matches, shared keys, no match). `review` flags are for the science lead (name-only matches, *Crocus sativus*, which WCVP lists as native although saffron is cultivated). Nothing is changed automatically; the flags travel into `species.json`.

In [9]:
fl = flags.run()
fl[(fl.data_quality.map(len) > 0) | (fl.review.map(len) > 0)].assign(
    data_quality=lambda d: d.data_quality.map(", ".join), review=lambda d: d.review.map(", ".join)
)[["taxon_name", "data_quality", "review", "records_attributed"]].head(20).reset_index(drop=True)

data_quality: {'gbif_synonym_match': 32, 'gbif_no_match': 11, 'gbif_shared_key': 4}
review: {'name_only_match': 50, 'native_status_check': 1}
records not attributed (shared key, GBIF-synonym twin): 2


,taxon_name,data_quality,review,records_attributed
0,Quercus frainetto,gbif_synonym_match,,True
1,Stachys angustifolia,,name_only_match,True
2,Trichogalium rotundifolium,gbif_synonym_match,,True
3,Carex ferruginea,,name_only_match,True
4,Carex praecox,,name_only_match,True
5,Libanotis pyrenaica,gbif_synonym_match,,True
6,Ligusticum lucidum,gbif_synonym_match,,True
7,Lathyrus clymenum,,name_only_match,True
8,Cyperus alopecuroides,,name_only_match,True
9,Mutarda nigra,gbif_synonym_match,,True


In [10]:
download = json.loads((config.WORK_DIR / "gbif_download.json").read_text())
print("GBIF occurrence download:", download)

GBIF occurrence download: {'download_key': '0000862-261104105008071', 'requested': '2026-11-04 23:58:41', 'n_taxon_keys': 1373, 'doi': '10.15468/dl.g7nrxe', 'status': 'SUCCEEDED', 'total_records': 333393}


*Phase 2 continues below: occurrences, grid cells, Natura 2000, fire, risk score, export.*